# 01-04 autograd 自动求导与最小线性回归

这一份只讲自动求导、梯度累加、no_grad、detach、item，以及用一个最小线性回归串起训练循环。


In [2]:
import torch
import numpy as np

torch.manual_seed(42)
device = torch.device("cuda")

print("torch version:", torch.__version__)
print("device:", device)


torch version: 2.10.0
device: cuda


## 13. 设备 device：CPU 和 GPU

Tensor 在哪里算，由 `device` 决定。

| 写法 | 作用 |
|---|---|
| `torch.device("cpu")` | 使用 CPU |
| `torch.device("cuda")` | 使用默认 GPU |
| `x.to(device(目标设备))` | 把 Tensor 移到指定设备 |
| `model.to(device(目标设备))` | 把模型参数移到指定设备 |

最常见报错：输入在 CPU，模型在 GPU，或者反过来。原则很简单：模型和数据必须在同一个设备上。

In [3]:
x = torch.randn(2, 3)
x = x.to(device)

print(x.device)

cuda:0


## 14. autograd 自动求导：先理解这 4 个词

| 词 | 人话解释 |
|---|---|
| `requires_grad=True` | 告诉 PyTorch：这个 Tensor 参与梯度计算 |
| `grad_fn` | 这个 Tensor 是通过什么运算得到的 |
| `loss.backward()` | 从 loss 开始反向传播，自动算梯度 |
| `x.grad` | loss 对 x 的梯度 |

在机器学习里你学过“求导、梯度下降”。PyTorch 的 autograd 就是帮你把链式法则自动做了。

In [4]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x + 1

print("y:", y)
print("y.grad_fn:", y.grad_fn)

y.backward()
print("x.grad:", x.grad)

y: tensor(11., grad_fn=<AddBackward0>)
y.grad_fn: <AddBackward0 object at 0x000001E389ABF9D0>
x.grad: tensor(7.)


手算检查：

$$y = x^2 + 3x + 1$$

$$\frac{dy}{dx} = 2x + 3$$

当 `x = 2` 时，梯度是 `7`，所以上面 `x.grad` 应该等于 `7`。

### 14.1 梯度会累加：为什么要 `zero_grad()`

PyTorch 默认会把每次 `backward()` 得到的梯度累加到 `.grad` 里，而不是自动覆盖。

所以训练模型时，每轮更新前必须先清空旧梯度：

```python
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

y1 = x ** 2
y1.backward()
print("第一次 backward 后:", x.grad)

y2 = x ** 2
y2.backward()
print("第二次 backward 后，梯度累加了:", x.grad)

x.grad.zero_()
print("手动清零后:", x.grad)

## 15. `detach()`、`no_grad()`、`item()`

| 方法 | 作用 | 常见场景 |
|---|---|---|
| `x.detach()` | 从计算图里分离出来，不再追踪梯度 | 保存预测结果、转 NumPy |
| `with torch.no_grad():` | 代码块内不记录梯度 | 验证、测试、推理 |
| `x.item()` | 把只有一个元素的 Tensor 转成 Python 数字 | 打印 loss |

人话：训练要梯度，评估不要梯度。不要把验证集、测试集也放进计算图里浪费内存。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2

print("y.requires_grad:", y.requires_grad)
print("y.detach().requires_grad:", y.detach().requires_grad)

loss = y.mean()
print("loss as tensor:", loss)
print("loss as Python number:", loss.item())

## 16. 最小线性回归：把前面的概念串起来

这里不堆代码，只看训练循环最小骨架。

任务：生成接近 `y = 3x + 2` 的数据，让模型学出权重和偏置。

你需要看懂这几个对象：

| 对象 | 作用 |
|---|---|
| `nn.Linear(1, 1)` | 线性模型，输入 1 个特征，输出 1 个值 |
| `nn.MSELoss()` | 均方误差，回归任务常用 |
| `torch.optim.SGD(...)` | 随机梯度下降优化器 |
| `model(x)` | 调用模型做前向计算 |
| `loss.backward()` | 自动计算参数梯度 |
| `optimizer.step()` | 根据梯度更新参数 |

In [ ]:
from torch import nn

torch.manual_seed(42)

x = torch.linspace(-3, 3, 200).reshape(-1, 1)  # start(起始值), end(终止值), steps(元素个数)
noise = 0.3 * torch.randn_like(x)
y = 3 * x + 2 + noise

model = nn.Linear(1, 1)  # in_features(输入特征数), out_features(输出特征数)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)  # params(模型参数), lr(学习率)

for epoch in range(100):
    pred = model(x)
    loss = loss_fn(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("learned weight:", model.weight.item())
print("learned bias:", model.bias.item())
print("final loss:", loss.item())

## 17. 第一节必须掌握的总结

### 17.1 创建 Tensor 时怎么选方法

| 你想做什么 | 推荐方法 |
|---|---|
| 从 Python 列表创建 | `torch.tensor(data(数据))` |
| 指定数据类型 | `torch.tensor(data(数据), dtype=torch.float32(元素类型))` |
| 创建全 0 | `torch.zeros(shape(形状))` |
| 创建全 1 | `torch.ones(shape(形状))` |
| 创建指定值 | `torch.full(shape(形状), value(填充值))` |
| 创建等差整数序列 | `torch.arange(start(起始值), end(终止值), step(步长))` |
| 创建等间隔浮点序列 | `torch.linspace(start(起始值), end(终止值), steps(元素个数))` |
| 创建 0 到 1 随机数 | `torch.rand(shape(形状))` |
| 创建正态分布随机数 | `torch.randn(shape(形状))` |
| 照着已有 Tensor 创建 | `torch.zeros_like(x)` / `torch.ones_like(x)` |

### 17.2 调试 Tensor 时先看什么

按这个顺序看：

1. `shape`：形状对不对。
2. `dtype`：类型对不对。
3. `device`：是不是都在 CPU 或都在 GPU。
4. `requires_grad`：该求导的有没有开，不该求导的有没有关。

### 17.3 训练循环为什么是那几步

1. `pred = model(x)`：前向传播，得到预测。
2. `loss = loss_fn(pred, y)`：计算预测和真实值的差距。
3. `optimizer.zero_grad()`：清空旧梯度。
4. `loss.backward()`：反向传播，计算新梯度。
5. `optimizer.step()`：更新参数。

## 18. 自检题

学完这一节，不要求你默写所有 API，但要能回答：

1. `torch.tensor([1, 2, 3])` 和 `torch.Tensor(2, 3)` 有什么区别？
2. `torch.rand` 和 `torch.randn` 的随机数分布有什么区别？
3. 为什么分类标签经常要转成 `long`？
4. `reshape` 和 `view` 的区别是什么？初学时优先用哪个？
5. `unsqueeze(0)` 和 `unsqueeze(1)` 对一维向量的形状分别有什么影响？
6. 为什么模型和输入 Tensor 必须在同一个 device 上？
7. 为什么每次训练前要 `optimizer.zero_grad()`？
8. `detach()` 和 `torch.no_grad()` 分别适合什么时候用？

## 19. 下一节怎么学

下一节再进入：

- `Dataset`：把样本组织起来。
- `DataLoader`：按 batch 读取样本。
- `nn.Module`：定义自己的神经网络。
- 二分类 MLP：从“会操作 Tensor”过渡到“会训练模型”。